In [ ]:
import pandas as pd
import seaborn as sb

dane = pd.read_csv("data4.csv")
fig_corr = sb.heatmap(dane.corr())



Mamy silną korelację pomiędzy zmiennymi x8, x9 a y oraz lekką między x2 i y. Zmienne nie są współliniowe, dzięki czemu możemy bezpiecznie użyć regresji liniowej. Pozostałe zmiennie nie są mocno skorelowane z wynikiem końcowym


In [ ]:
import matplotlib.pyplot as plt

dane.plot(kind="box",subplots=True,layout=(5,5),figsize=(15,15))
plt.tight_layout()
plt.show()


Wszystkie wartości xi mieszczą sie mniej więcej w przedziale [-3,3] oprócz jednego outliera w x15, ale ma on mały wpływ na końcowy wynik, przez co można pominąć krok odrzucania outlierów

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import Ridge,LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler


X=dane.drop(columns="y")
y=dane["y"]


X_temp,X_test,y_temp,y_test = train_test_split(X,y,test_size=0.2,random_state=42)
X_train,X_val,y_train,y_val = train_test_split(X_temp,y_temp,test_size=0.25,random_state=42)

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)


lr = LinearRegression()

lr.fit(X_train_scaled,y_train)
y_pred_lr = lr.predict(X_test_scaled)
mse_lr = mean_squared_error(y_test,y_pred_lr)

alphas = [0.01,0.1,1,10,500]
best_alpha = 0
lowest_mse = 99999999

for alpha in alphas:
    l2 = Ridge(alpha=alpha)
    l2.fit(X_train_scaled,y_train)
    
    y_pred_val= l2.predict(X_val_scaled)
    mse_val = mean_squared_error(y_val,y_pred_val)
    
    print(f"MSE value for alpha = {alpha}: {mse_val}")
    if(mse_val < lowest_mse):
        best_alpha=alpha
        lowest_mse=mse_val

print(f"Best MSE value is {lowest_mse} for alpha={best_alpha}")

l2_best = Ridge(alpha=best_alpha)
l2_best.fit(X_train_scaled,y_train)
y_pred_l2 = l2_best.predict(X_test_scaled)

mse_l2 = mean_squared_error(y_test,y_pred_l2)

table_data={
    "Model" : ["Regresja Liniowa ","Regresja L2"],
    "Błąd MSE" : [mse_lr,mse_l2]
}
table = pd.DataFrame(table_data)

print(table)

print("Wagi Regresji Liniowej:", lr.coef_)
print("Wagi Regresji L2:", l2_best.coef_)
    

Widzimy, że w tym wypadku wykorzystanie modelu, który premiuje niskie wagi nie zrobiło dużej różnicy, ponieważ najlepszy wynik był dla bardzo małej kary za te wysokie wagi. Spowodowane jest to również tym, że dane xi były między sobą słabo skorelowane co nie wymagało od zwykłej regresji liniowej windowania wag do wysokich poziomów.